### weight map

In [1]:
import torch 

state_dict = torch.load(
    "pytorch_model.bin",
    map_location='cpu',
    weights_only= True,
    mmap= True
)

In [2]:
import json

with open("modelWeighStructure.json", "w") as f:
    model_structure = {
        layer_name: state_dict[layer_name].shape for layer_name in state_dict.keys()
    }

    json.dump(model_structure, f, indent=4)

In [3]:
for layer, shape in json.load(open("modelWeighStructure.json")).items():
    if 'model.encoder.layers.0.' in layer:
        print(f"Name: {layer} | Shape: {shape}")

Name: model.encoder.layers.0.self_attn.k_proj.weight | Shape: [1024, 1024]
Name: model.encoder.layers.0.self_attn.k_proj.bias | Shape: [1024]
Name: model.encoder.layers.0.self_attn.v_proj.weight | Shape: [1024, 1024]
Name: model.encoder.layers.0.self_attn.v_proj.bias | Shape: [1024]
Name: model.encoder.layers.0.self_attn.q_proj.weight | Shape: [1024, 1024]
Name: model.encoder.layers.0.self_attn.q_proj.bias | Shape: [1024]
Name: model.encoder.layers.0.self_attn.out_proj.weight | Shape: [1024, 1024]
Name: model.encoder.layers.0.self_attn.out_proj.bias | Shape: [1024]
Name: model.encoder.layers.0.self_attn_layer_norm.weight | Shape: [1024]
Name: model.encoder.layers.0.self_attn_layer_norm.bias | Shape: [1024]
Name: model.encoder.layers.0.fc1.weight | Shape: [8192, 1024]
Name: model.encoder.layers.0.fc1.bias | Shape: [8192]
Name: model.encoder.layers.0.fc2.weight | Shape: [1024, 8192]
Name: model.encoder.layers.0.fc2.bias | Shape: [1024]
Name: model.encoder.layers.0.final_layer_norm.weight

## Rebuilding model
### Encoder

In [12]:
class MultiHeadAtention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_attn_head = config["encoder_attention_heads"]
        self.attn_dim = self.d_model // self.num_attn_head

        self.k_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight
        self.k_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.k_proj.weight"]
        )
        self.q_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.q_proj.weight"]
        )
        self.v_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.v_proj.weight"]
        )
        self.out_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.out_proj.weight"]
        )

        # bias
        self.k_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.k_proj.bias"]
        )
        self.q_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.q_proj.bias"]
        )
        self.v_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.v_proj.bias"]
        )
        self.out_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.out_proj.bias"]
        )

    @torch.no_grad()
    def forward(self, x):
        B, S, _ = x.shape

        k = self.k_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        q = self.q_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)

        attn_scores = k @ q.transpose(-2, -1) * (self.attn_dim**-0.5)
        attn_scores = torch.nn.functional.softmax(attn_scores, dim=-1)

        out = attn_scores @ v
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        out = self.out_proj(out)

        return out

In [16]:
class EncoderFeedForwardNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()

        self.d_model = config["d_model"]
        self.ffn_dim = config["encoder_ffn_dim"]

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.ffn_dim)
        self.act = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(in_features=self.ffn_dim, out_features=self.d_model)

        # weight
        self.fc1.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc1.weight"]
        )
        self.fc2.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc2.weight"]
        )

        # bias
        self.fc1.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc1.bias"]
        )
        self.fc2.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc2.bias"]
        )

    @torch.no_grad()
    def forward(self, x):

        return self.fc2(self.act(self.fc1(x)))

In [15]:
model_config = json.load(open("config.json"))
dummy_attn_input = torch.randn(4, 12, model_config['d_model'])

mha = MultiHeadAtention(model_config, 0, state_dict)
effn = EncoderFeedForwardNetwork(model_config, 0, state_dict)
encoder_ffn_output = effn(mha(dummy_attn_input))
encoder_ffn_output.shape

torch.Size([4, 12, 1024])